# Case File: The Broken Reference
A report can still point to a field or measure that no longer exists in its semantic model.
In this investigation, **SFCaseFiles_PNW_Lineage** references a missing measure named **Incident Count**. We will find that reference, replace it with **Total Incidents**, and check the result.
Open the report's **Incident Trend** page first and look at the broken table visual. Our path is: **inspect → repair → verify**.

---
## Case File #0 — Stocking the Evidence Room
This notebook keeps its setup here so the investigation can use the configuration below
without calling a separate setup notebook.
**Before the demo:** prepare Semantic Link Labs **0.18.1** in this notebook's
session or attached Fabric Environment. Run the optional install cell only if needed.
**Changing targets:** edit the model names, report (where applicable), and workspace
in the configuration cell. A different schema may also require changes in the investigation cells.
**Runtime:** This investigation does not use Spark operations or require a default Lakehouse. The example uses a PBIR report; repair actions update that report.

### Inline setup used in this demo
Run the configuration and imports below. The installation cell is optional preparation.
If installation requests a session restart, restart before running the configuration and imports.

In [ ]:
# Optional preparation: run this cell only if the pinned package is not already installed.
%pip install semantic-link-labs==0.18.1 --quiet

In [ ]:
# Configuration: change the investigation targets here.
WORKSPACE = None  # None = the current Fabric workspace
DATASET_REPAIR = "SFCaseFiles"
REPORT_REPAIR = "SFCaseFiles_PNW_Lineage"

In [ ]:
import importlib.metadata
import notebookutils
import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd
from sempy_labs.semantic_model import lineage_view
from sempy_labs.report import ReportWrapper

sll_version = importlib.metadata.version("semantic-link-labs")
print(f"Semantic Link Labs: {sll_version} (demo target: 0.18.1)")

context = notebookutils.runtime.context
WORKSPACE_ID = WORKSPACE or context["currentWorkspaceId"]
WORKSPACE_NAME = context["currentWorkspaceName"]
LAKEHOUSE_ID = context.get("defaultLakehouseId")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

print(f"Notebook workspace: {WORKSPACE_NAME}")
print(f"Model workspace: {WORKSPACE or WORKSPACE_NAME}")
print(f"Main model: {DATASET_REPAIR}")
print(f"Report: {REPORT_REPAIR}")

In [ ]:
# See the model's measures and the intended replacement.
#measures = fabric.list_measures(dataset=DATASET_REPAIR, workspace=WORKSPACE)
#display(measures)

## 1. Find the broken reference
Read the objects used by the report and compare them with the semantic model. With `extended=True`, the result includes **Valid Semantic Model Object**.
Rows marked `False` identify references that no longer resolve. The count is a count of returned references, not necessarily a count of broken visuals.

In [ ]:
# Inspect the objects referenced by this report.
report = ReportWrapper(report=REPORT_REPAIR, workspace=WORKSPACE)
objects_before = report.list_semantic_model_objects(extended=True)

# Find references that no longer match the semantic model.
broken_before = objects_before[
    objects_before["Valid Semantic Model Object"].eq(False)
]

display(broken_before)
print(f"Invalid references before repair: {len(broken_before)}")

## 2. Follow the trail and repair the reference
In the lineage view, select **SFCaseFiles_PNW_Lineage** and analyze its references.
Find the missing **Incident Count** measure, choose **Total Incidents** as its replacement, and save the change. This updates the report.
A replacement must answer the intended business question. An object existing in the model does not, by itself, make it the right replacement.

In [ ]:
lineage_view(dataset=DATASET_REPAIR, workspace=WORKSPACE, dark_mode=True)

## 3. Verify the repair
After saving in the lineage view, read the report definition again and repeat the reference check.
Then reopen or refresh the report's **Incident Trend** page. Confirm that the table visual works and that its values match the intended **Total Incidents** measure. A valid reference is one part of verification; the result must also make sense.

In [ ]:
# Reload the saved report so we inspect the latest definition.
report = ReportWrapper(report=REPORT_REPAIR, workspace=WORKSPACE)
objects_after = report.list_semantic_model_objects(extended=True)

broken_after = objects_after[
    objects_after["Valid Semantic Model Object"].eq(False)
]

display(broken_after)
print(f"Invalid references: {len(broken_before)} before, {len(broken_after)} after")

validity = objects_after["Valid Semantic Model Object"]
if objects_after.empty or validity.isna().any():
    print("Reference check is incomplete. Review the diagnostics before drawing a conclusion.")
elif not validity.isin([True, False]).all():
    print("Reference check returned unexpected values. Review the diagnostics.")
elif broken_after.empty:
    print("No invalid references found. Refresh Incident Trend and verify the visual's values.")
else:
    print("Invalid references remain. Review them in the lineage view.")

## What we learned
We used report metadata to find a missing measure, followed the model's downstream lineage, and repaired the report reference without changing the semantic model.
For another report, change `DATASET_REPAIR` and `REPORT_REPAIR`, inspect its invalid references, and choose replacements based on their business meaning.
### Resources
- [ReportWrapper examples](https://github.com/microsoft/semantic-link-labs/wiki/Code-Examples#find-broken-visuals-in-a-power-bi-report)
- [Semantic Link Labs lineage view](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.semantic_model.html#sempy_labs.semantic_model.lineage_view)